In [3]:
import numpy as np 
#need to convert the cellml file for the Trovato model into a python version that can be used instead 

#will set up initialisation values later - need to make a list of those that need initialising
def Cell_Geometry(L = 0.0164, rad=0.00175):
    #returns 9 values 
    #this function returns all cell geometry values needed later in the model
    #both L and rad are in cm 
    V_cell = 1000 * np.pi * rad**2 * L #cell volume in cm^3
    A_geo = 2 * np.pi * rad**2 + 2 * np.pi * rad * L #cell surface area in cm^2 
    A_cap = 2 * A_geo #find out what this represents
    V_myo = 0.6 * V_cell
    V_nsr = 0.04 * V_cell
    V_jsr = 0.002 * V_cell
    V_csr = 0.008 * V_cell
    V_ss = 0.02 * V_cell
    V_sl = 0.15 * V_cell
    return V_cell, A_geo, A_cap, V_myo, V_nsr, V_jsr, V_csr, V_ss, V_sl

def If(v, time, ENa, EK, GfNa = 0.0116, GfK = 0.0232, y = 0.233119011214908):
    #This function calculates the current from the fast sodium and potassium channels
    #v is the membrane potential in mV
    #time is the time in ms
    yss = 1/(1 + np.exp(v + 87)/9.5)
    tauy = 2000 / (np.exp((v + 57)/60) + np.exp(-(v + 132)/10))
    dy_dt = (yss - y)/tauy
    IfNa = GfNa * y **2 * (v-ENa)
    IfK = GfK * y**2 * (v-EK)
    If = IfNa + IfK
    return If, dy_dt

def Membrane_Potential(R,T,F, I_Na, I_NaL, I_to, I_sus, I_CaL, I_CaT, I_CaNa, I_CaK, I_Kr, I_Ks, I_f, I_K1, I_NaCa_i, I_NaCa_ss, I_NaK, I_Nab, I_pCa, I_Cab):
    #set initial values
    amp = -40 #microA per microF
    v = -86.6814002878592 #mV
    duration = 1 #ms
    if time <= duration:
        I_stim = amp
    else:
        I_stim = 0
    vffrt = v * F**2 / (R*T)
    vfrt = v * F / (R*T)
    dv_dt = -(I_Na + I_NaL + I_to + I_sus + I_CaL + I_CaT + I_CaNa + I_CaK + I_Kr + I_Ks + I_f + I_K1 + I_NaCa_i + I_NaCa_ss + I_NaK + I_Nab + I_pCa + I_Cab + I_stim)
    return dv_dt, vffrt, vfrt

def CaMK(cass, time):
    #initial values
    KmCaMK = 0.15 #mM
    aCaMK = 0.05 # per mM per ms
    bCaMK = 0.00068 #per ms
    CaMKo = 0.05 #per ms
    KmCaM = 0.0015 #mM
    CaMKt = 0.00505983330678751 #mM
    
    CaMKb = CaMKo * (1 - CaMKt)/(1 + (KmCaM/cass))
    CaMKa = CaMKb + CaMKt

    dCaMKa_dt = aCaMK * CaMKb * (CaMKb + CaMKt) - bCaMK * CaMKt

    return CaMKb, CaMKa, dCaMKa_dt

def Reverse_Potentials(R,T,F, nao, ko, cao, nasl, ksl, casl):
    #initial values
    PKNa = 0.01833
    ENa = (R*T/F) * np.log(nao/nasl)
    EK = (R*T/F) * np.log(ko/ksl)
    ECa = (0.5 * R*T/F) * np.log(cao/casl)
    EKs = (R*T/F) * np.log((ko + PKNa * nao)/(ksl + PKNa * nasl))

    return ENa, EK, ECa, EKs

def INa(KmCaMK, v, CaMKa, ENa, time):
    #initial values
    # INa component: parameters and initial state values

    # Voltage-dependence parameters for activation gate m
    mssV1 = 48.4264
    mssV2 = 7.5653
    mtV1 = 11.64
    mtV2 = 34.77
    mtD1 = 6.765
    mtD2 = 8.552
    mtV3 = 77.42
    mtV4 = 5.955

    # Initial value of activation gate m
    m0 = 0.00632661703915808

    # Voltage-dependence parameters for inactivation gates
    hssV1 = 78.5
    hssV2 = 6.22

    # Initial value for the fast inactivation gate
    Ahf = 0.99
    hf0 = 0.788611739889677

    # Initial value for the slow inactivation gate
    hs0 = 0.788545979951331

    # Maximum sodium conductance
    GNa = 39.4572

    # Initial value for the j inactivation gate
    j0 = 0.790474358603666

    # Initial value for phosphorylated fast inactivation gate
    hsp0 = 0.579693514309867

    # Initial value for phosphorylated j gate
    jp0 = 0.790947058236417

    mss = 1/ (1 + np.exp((-v-mssV1)/mssV2))
    tm = 1 / (mtD1 * np.exp((v + mtV1)/mtV2 + mtD2 * np.exp(-(v + mtV3)/mtV4)))
    dm_dt = (mss - m0)/tm
    hsss = 1 / (1 + np.exp((v + hssV1)/hssV2))
    thf = 1 / (3.686 * 10 **-6 * np.exp(-(v + 3.8875)/7.8579) + 16 * np.exp((v - 0.4963)/9.1843))
    ths = 1 / (0.-0.009794 * np.exp(-(v + 17.95)/28.05) + 0.3343 * np.exp((v+ 5.73)/56.66))
    Ahs = 1 - Ahf
    dhf_dt = (hss-hf)/thf
    dhs_dt = (hss - hs)/ths
    h = Ahf * hf + Ahs * hs
    jss = hss
    tj = 4.859 + 1/ (0.8628 * np.exp(-(v + 116.728)/7.6005) + 1.1096 * np.exp((v + 6.2719)/9.0358))
    dj_dt = (jss - j)/tj
    hssp = 1 / (1 + np.exp((v + 84.7)/6.22))

    thsp = 3 * ths
    dhsp_dt = (hssp - hsp)/thsp
    hp = Ahf * hf + Ahs * hsp
    tjp = 1.46 * tj 
    djp_dt = (jss - jp)/tjp
    fINaP = 1/(1 + KmCaMK/CaMKa)
    INa = GNa * (v - ENa) * m**3 * ((1 - fINaP) * h * j + fINaP * hp * jp)

    return INa, tm


